## 🪄 二、三步出图：Seaborn 极速实战

### 📌 第 1 步：导入库 + 加载数据

这次我们使用 Seaborn 内置的经典数据集 —— **`healthexp`**！

In [ ]:
# 📦 1. 环境准备与数据加载
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 通过 rc 参数设置中文字体
sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 加载 healthexp 数据集：各国医疗支出数据
df = sns.load_dataset("healthexp")

# 快速查看数据结构
print(df.head())
print(f"\n数据规模：{df.shape}")
print(f"\n国家列表：{df['Country'].unique()}")

> **💡 数据集简介：**
> `healthexp` 收录了 **6 个发达国家从 1970 到 2020 年的医疗支出与预期寿命数据**，包含 4 个字段：
> - `Country`：国家名称
> - `Year`：年份（1970-2020，每 5 年一条）
> - `Spending_USD`：人均医疗支出（美元）
> - `Life_Expectancy`：预期寿命（年）
>
> 这是理解累积分布与跨国对比的经典数据集！

### 📌 第 2 步：一行代码画出累积分布

运营总监要看各国医疗支出的分布集中度，Seaborn 怎么做？

In [ ]:
# 📈 2. 基础累积分布图
fig, ax = plt.subplots(figsize=(10, 6))

# Seaborn 一行代码画出 ECDF
sns.ecdfplot(
    data=df,
    x="Spending_USD",  # x 轴：医疗支出
    ax=ax,
    linewidth=2,       # 线条粗细
    color="#2c3e50"    # 深蓝灰色
)

# Matplotlib 精细美化
ax.set_title("各国人均医疗支出累积分布", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("人均医疗支出（美元）", fontsize=12)
ax.set_ylabel("累积比例", fontsize=12)

# 添加 80/20 参考线
ax.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="80% 阈值")
ax.axvline(x=df["Spending_USD"].quantile(0.8), color="red", linestyle="--", alpha=0.7, linewidth=1.5)

# 标注拐点数值
p80_value = df["Spending_USD"].quantile(0.8)
ax.text(p80_value + 100, 0.82, f"80% 分位点: ${p80_value:.0f}", 
        fontsize=11, color="red", fontweight="bold")

ax.legend(loc="lower right", fontsize=10)

plt.tight_layout()
plt.savefig("ecdf_basic.png", dpi=300, bbox_inches="tight")
print("✅ 累积分布图已保存为 'ecdf_basic.png'")
plt.show()

**✅ 完成！**

你没有看错，核心代码就这几行：
* `sns.ecdfplot()` 是 Seaborn 的累积分布图入口
* `x="Spending_USD"` 指定要分析分布的数值列
* `ax.axhline(y=0.8)` 画 80% 水平参考线
* `ax.axvline(x=quantile)` 画对应的支出阈值线
* 交点就是 **80/20 拐点**！

> **🤔 什么是 ECDF（经验累积分布函数）？**
> 简单理解：**曲线告诉你"有多少比例的数据小于等于某个值"**。
>
> 比如曲线在 $3000 处对应 y=0.75，意味着 75% 的国家-年份数据人均支出低于 $3000。
> 
> 相比直方图，ECDF 的优势：**不需要调 bin 大小，不会丢失信息，拐点精准可读。**

### 📌 第 3 步：双剑合璧——多国对比 + 帕累托诊断

运营总监得寸进尺了："能不能把 6 个国家画在同一张图上？我想看谁的医疗支出集中度最高！"

没问题，`ecdfplot` 支持 **`hue` 参数叠加多维度曲线**！

In [ ]:
# 🚀 3. 高级玩法:多国对比累积分布
fig, ax = plt.subplots(figsize=(12, 7))

# 自定义国家颜色映射（让颜色与国家强绑定）
country_colors = {
    "USA":                "#e74c3c",  # 红色 - 美国
    "Germany":            "#f39c12",  # 橙色 - 德国
    "Japan":              "#3498db",  # 蓝色 - 日本
    "France":             "#2ecc71",  # 绿色 - 法国
    "Great Britain":      "#9b59b6",  # 紫色 - 英国
    "Canada":             "#1abc9c"   # 青色 - 加拿大
}

# 按国家分组画累积分布曲线
sns.ecdfplot(
    data=df,
    x="Spending_USD",
    hue="Country",       # 🔥 核心！按国家分组
    hue_order=sorted(df["Country"].unique()),
    linewidth=2.5,
    ax=ax,
    palette=country_colors
)

# Matplotlib 注入帕累托基准线
ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.6, linewidth=1.2, label="80% 阈值线")

ax.set_title("六国人均医疗支出累积分布对比（寻找帕累托拐点）",
             fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("人均医疗支出（美元）", fontsize=12)
ax.set_ylabel("累积比例", fontsize=12)

# 🔥 核心优化：图例外置 + 颜色-国家对照清晰
ax.legend(title="国家", bbox_to_anchor=(1.02, 1), loc="upper left", 
          fontsize=11, title_fontsize=12, framealpha=0.9)

# 🔥 在曲线末端直接标注国家名称（颜色与线条一致）
for country, color in country_colors.items():
    country_data = df[df["Country"] == country]["Spending_USD"].dropna().sort_values()
    if len(country_data) > 0:
        max_val = country_data.iloc[-1]
        ax.annotate(country, xy=(max_val, 1.0), xytext=(max_val + 150, 0.95),
                   fontsize=11, fontweight="bold", color=color,
                   arrowprops=dict(arrowstyle="->", color=color, lw=1.5))

# 调整右侧边距，为图例和标注留出空间
plt.subplots_adjust(right=0.78)

# 网格线优化
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("ecdf_multi_country.png", dpi=300, bbox_inches="tight")
print("✅ 多国对比累积分布图已保存为 'ecdf_multi_country.png'")
plt.show()

**✅ 搞定！** 现在图表变成了多维度对比：
* **6 条曲线** = 每个国家一条累积分布线
* **颜色区分** = `hue="Country"` 自动着色
* **80% 基准线** = 灰色虚线横穿，快速判断拐点
* **曲线越陡** = 支出越集中；**曲线越缓** = 支出越分散

> **💡 如何解读这张图？**
> * 如果某国曲线在低支出区域快速上升到 0.8 → 该国医疗支出**高度集中**，大部分年份花在较低水平
> * 如果某国曲线缓慢爬升到 0.8 → 该国医疗支出**分布分散**，高低年份差异大
> * **对比各国曲线斜率** = 一眼看出谁的医疗支出波动更大

## 🎨 三、进阶玩法：自定义阈值 + 业务场景实战

### 📌 场景 1：电商平台用户消费 80/20 分析

In [ ]:
# 🛒 4. 电商场景:用户消费帕累托分析

# 模拟电商用户消费数据（实际业务中从数据库读取）
np.random.seed(42)
n_users = 1000

# 模拟帕累托分布：20% 用户贡献 80% 消费
user_ids = range(n_users)
# 用对数正态分布模拟真实的消费集中度
spendings = np.random.lognormal(mean=5, sigma=1.5, size=n_users)

ecommerce_df = pd.DataFrame({
    "user_id": user_ids,
    "spending": np.round(spendings, 2)
})

# 排序并计算累积占比
ecommerce_df = ecommerce_df.sort_values("spending", ascending=False).reset_index(drop=True)
ecommerce_df["cumulative_pct"] = (ecommerce_df.index + 1) / len(ecommerce_df)
ecommerce_df["cumulative_spend"] = ecommerce_df["spending"].cumsum()
ecommerce_df["cumulative_spend_pct"] = ecommerce_df["cumulative_spend"] / ecommerce_df["spending"].sum()

# 找到 80% 消费额对应的用户比例
top_users_pct = ecommerce_df[ecommerce_df["cumulative_spend_pct"] >= 0.8]["cumulative_pct"].iloc[0]
print(f"📊 帕累托分析结果：")
print(f"   前 {top_users_pct*100:.1f}% 的用户贡献了 80% 的消费额")
print(f"   即前 {int(top_users_pct * n_users)} 名用户")

In [ ]:
# 🛒 画出电商用户累积分布图
fig, ax = plt.subplots(figsize=(10, 6))

sns.ecdfplot(
    data=ecommerce_df,
    x="spending",
    ax=ax,
    linewidth=2.5,
    color="#3498db"
)

# 标注 80/20 拐点
ax.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, linewidth=1.5)
ax.axvline(x=ecommerce_df["spending"].quantile(0.8), color="red", linestyle="--", alpha=0.7, linewidth=1.5)

# 在曲线交点处添加标注
p80_spend = ecommerce_df["spending"].quantile(0.8)
ax.plot(p80_spend, 0.8, "ro", markersize=10)
ax.annotate(f"80% 用户\n消费 ≤ ${p80_spend:.0f}", 
            xy=(p80_spend, 0.8), 
            xytext=(p80_spend + 50, 0.65),
            fontsize=11, color="red", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="red", lw=2))

ax.set_title("电商平台用户消费额累积分布（帕累托诊断）",
             fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("用户消费额（美元）", fontsize=12)
ax.set_ylabel("累积用户比例", fontsize=12)

ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ecdf_ecommerce.png", dpi=300, bbox_inches="tight")
print("✅ 电商用户累积分布图已保存为 'ecdf_ecommerce.png'")
plt.show()

**✅ 效果拉满！** 业务洞察一目了然：
* **红色交点** = 80% 用户消费阈值，精准定位"长尾用户"边界
* **曲线陡峭** = 消费集中度高，少数大户贡献大部分收入
* **曲线平缓** = 消费分散，用户价值相对均匀

> **🤫 业务洞察：**
> 如果前 20% 用户贡献了超过 80% 的消费 → 你的平台**严重依赖高价值用户**
>
> 这意味着：
> - 流失几个大户会对收入造成巨大冲击
> - 需要建立用户分层运营体系
> - 中尾部用户有巨大增长潜力
>
> **图表只是第一步，后续还需要 cohort 分析和留存率追踪来背书。**

### 📌 场景 2：用底层 API 实现多阈值叠加

In [ ]:
# 🎯 5. 多阈值诊断:同时标注 50%/80%/95% 分位点
fig, ax = plt.subplots(figsize=(10, 6))

sns.ecdfplot(
    data=df,
    x="Spending_USD",
    ax=ax,
    linewidth=2.5,
    color="#2c3e50"
)

# 添加多个阈值线
thresholds = [0.5, 0.8, 0.95]
colors = ["#2ecc71", "#e74c3c", "#9b59b6"]

for pct, color in zip(thresholds, colors):
    value = df["Spending_USD"].quantile(pct)
    ax.axhline(y=pct, color=color, linestyle="--", alpha=0.5, linewidth=1)
    ax.axvline(x=value, color=color, linestyle="--", alpha=0.5, linewidth=1)
    ax.plot(value, pct, "o", color=color, markersize=8)
    ax.text(value + 100, pct + 0.03, f"{pct*100:.0f}%: ${value:.0f}", 
            fontsize=10, color=color, fontweight="bold")

ax.set_title("医疗支出多分位点诊断图", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("人均医疗支出（美元）", fontsize=12)
ax.set_ylabel("累积比例", fontsize=12)

ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ecdf_multi_threshold.png", dpi=300, bbox_inches="tight")
print("✅ 多阈值诊断图已保存为 'ecdf_multi_threshold.png'")
plt.show()

**✅ 效果：**
* **50% 中位线（绿色）** = 典型支出水平
* **80% 帕累托线（红色）** = 关键拐点
* **95% 极端值线（紫色）** = 异常高支出的边界
* **三个点连起来** = 完整描绘数据分布轮廓

## 📦 五、完整代码模板（复制即用）

下面是可直接复用的完整模板，替换数据和列名即可套用你的业务场景：

In [ ]:
# 💾 完整代码模板:累积分布图 + 帕累托诊断
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 加载数据（可替换为你自己的 CSV）
# df = pd.read_csv("your_data.csv")
df = sns.load_dataset("healthexp")

# 自定义颜色映射（颜色与类别强绑定，方便识别）
country_colors = {
    "USA":                "#e74c3c",  # 红色 - 美国
    "Germany":            "#f39c12",  # 橙色 - 德国
    "Japan":              "#3498db",  # 蓝色 - 日本
    "France":             "#2ecc71",  # 绿色 - 法国
    "Great Britain":      "#9b59b6",  # 紫色 - 英国
    "Canada":             "#1abc9c"   # 青色 - 加拿大
}

# 画累积分布图
fig, ax = plt.subplots(figsize=(10, 6))

sns.ecdfplot(
    data=df,
    x="Spending_USD",    # 要分析分布的数值列
    hue="Country",        # 可选：按某列分组
    hue_order=sorted(df["Country"].unique()),
    linewidth=2.5,
    ax=ax,
    palette=country_colors
)

# Matplotlib 注入帕累托基准线
ax.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="80% 阈值")

# 标注 80% 分位点
p80 = df["Spending_USD"].quantile(0.8)
ax.axvline(x=p80, color="red", linestyle="--", alpha=0.7, linewidth=1.5)
ax.plot(p80, 0.8, "ro", markersize=10)
ax.text(p80 + 100, 0.82, f"80% 分位点: ${p80:.0f}", 
        fontsize=11, color="red", fontweight="bold")

ax.set_title("累积分布图（帕累托诊断）", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("人均医疗支出（美元）", fontsize=12)
ax.set_ylabel("累积比例", fontsize=12)

# 🔥 核心优化：图例外置 + 颜色-国家对照清晰
ax.legend(title="国家", bbox_to_anchor=(1.02, 1), loc="upper left", 
          fontsize=11, title_fontsize=12, framealpha=0.9)

# 🔥 在曲线末端直接标注国家名称（颜色与线条一致）
for country, color in country_colors.items():
    country_data = df[df["Country"] == country]["Spending_USD"].dropna().sort_values()
    if len(country_data) > 0:
        max_val = country_data.iloc[-1]
        ax.annotate(country, xy=(max_val, 1.0), xytext=(max_val + 150, 0.95),
                   fontsize=11, fontweight="bold", color=color,
                   arrowprops=dict(arrowstyle="->", color=color, lw=1.5))

# 导出高清图
plt.tight_layout()
fig.savefig("ecdf_dashboard.png", dpi=300, bbox_inches="tight")
print("✅ 图表已保存为 'ecdf_dashboard.png'")
plt.show()

## 💡 六、本节核心总结

| 步骤 | 工具 | 核心 API | 作用 |
|------|------|----------|------|
| 基础 ECDF | Seaborn | `sns.ecdfplot(x=xxx)` | 一行代码画累积分布 |
| 多维度对比 | Seaborn | `hue=xxx` | 按类别叠加多条曲线 |
| 帕累托基准线 | Matplotlib | `ax.axhline(y=0.8)` | 注入 80% 参考线 |
| 分位点标注 | Matplotlib | `df.quantile(0.8)` | 精准定位拐点 |
| 多阈值诊断 | Matplotlib | `ax.axvline()` + `ax.annotate()` | 叠加 50/80/95 分位 |
| 极值处理 | Pandas | `df.dropna()` / `df.quantile(0.99)` | 清理异常数据 |

**给你的练习建议：**
1. 📝 换成 `sns.load_dataset("penguins")`，用 `ecdfplot(x="body_mass_g", hue="species")` 分析企鹅体重分布。
2. 🎨 把 `palette="Set2"` 改成 `"viridis"`, `"pastel"`, `"colorblind"`，感受多曲线配色差异。
3. 🔄 试试 `ax.set_xscale("log")` 将 x 轴改为对数坐标，观察曲线形态变化。
4. 📊 用你自己的业务数据（如用户消费、订单金额、访问时长），画出帕累托诊断图，找到你的 80/20 拐点。